# 杠杆ETF份额快照与口径门
SOXL/SOXS公开原始CSV；保留TradeDate与获取时间，不用不同日期NAV拼接AUM，不将互换名义敞口之和当作净资产。只建立观察时点缓存，不生成交易信号。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/leveraged_fund_snapshot_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.leveraged_fund_snapshot import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'leveraged_fund_snapshot_20261014',
 'plan_sha256': 'b10c295b4afce93f0eb86066088503044bfb40dfad03454443100a4d92ff29c0',
 'decision': 'needs_historical_flow_and_publication_data',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'funds': 2,
 'distinct_reported_trade_dates': 1,
 'historical_training_eligible': False,
 'intraday_flow_identified': False,
 'net_assets_from_holdings_sum_allowed': False,
 'exact_shares_authority': 'original CSV header and Recorder metadata; native shares_million is approximate float32 cache, not a share ledger',
 'snapshot_summary': [{'symbol': 'SOXL',
   'reported_trade_date': '2026-10-13',
   'shares_outstanding_exact': 162000060,
   'observed_at_utc': '2026-10-13T20:48:18.490708+00:00',
   'cache_available_at_ny': '2026-10-13T16:49:00-04:00',
   'holdings_rows': 43,
   'swap_asset_rows': 9,
   'reported_market_value_sum': 69526649114.19,
   'reported_holdings_percent_sum': 334.6033933151999,
   'market_value_sum_is_net_ass

# 杠杆ETF份额：已取得原始快照，资金流历史仍不足

0拟合、0回测。通过发行人公开页面实际显示的链接取得[SOXL原始CSV](https://www.direxion.com/holdings/SOXL.csv)与[SOXS原始CSV](https://www.direxion.com/holdings/SOXS.csv)，两文件均HTTP200，原始字节和获取时间已冻结到NAS，不把浏览器显示误称为原始文件。

产品页普通HTTP返回403；Kimi WebBridge无扩展连接，内置浏览器不可用，随后通过可用Chrome正常公开页面打开持仓菜单。浏览器下载等待未返回文件，使用该菜单中实际观察到的公开CSV链接完成下载；未猜测历史URL、导出登录凭据或绕过访问控制。读取原始CSV与可见表格份额一致。

|基金|CSV TradeDate|精确流通份额|持仓行|互换资产腿行|原始比例合计|
|---|---|---:|---:|---:|---:|
|SOXL|2026-10-13|162,000,060|43|9|334.6034%|
|SOXS|2026-10-13|65,568,272|12|8|-194.1204%|

页面两位小数显示合计约334.58%/-194.13%，与CSV高精度逐项舍入后的差异正常。列内包括互换资产腿敞口，直接加总MarketValue不能当净资产；负的SOXS合计也不是负NAV。没有据此构造AUM或再平衡金额。

页面可见NAV为SOXL128.26、SOXS37.26，标注日期2026-10-12；CSV的TradeDate是10月13日。两种日期定义尚未核实一致，不能无条件相乘作为同口径总资产。CSV的午夜TradeDate也不是发布时间。

本机首次成功取得原始CSV在2026-10-13 20:48:18 UTC左右，即纽约16:48之后；Qlib观察缓存使用向上取整到纽约16:49的可用时刻，绝不倒填当日开盘或收盘前。精确份额保存在原始头部和Recorder整数metadata；原生`shares_million`是float32特征缓存，不是份额账本，不能直接以有损缓存差分认定微小申赎。当前研究价格截止10月9日，这份后来取得的快照不进入那些历史训练。

## 现有分钟数据不能补出份额

对固定NAS源的SOXL780060条原始分钟记录核查：turnover_rate全部为0，777756条有正成交量却没有正换手率。因此该字段不能作为成交量反推历史流通份额的分母；这是不可用字段，不是零交易或零资金流。原始值不改写，源manifest和Parquet哈希保留在summary/Recorder。

机制依据：[联储2014年研究](https://www.federalreserve.gov/econres/feds/are-concerns-about-leveraged-etfs-overblown.htm)指出，资金流会改变甚至抵消杠杆ETF机械再平衡需求。因此只有价格与杠杆倍数，并不构成已观察的净买卖压力。此文不是SOXL稳定超额的证明。

决定：**needs_historical_flow_and_publication_data**。现在仅有一个TradeDate的两基金快照，尚无可用的连续份额/净资产版本、拆股映射及申赎生效/披露时点；日内净申赎保持未知，不填0。不能将已失败的SOXX份额H5路线替换标签后当作SOXL/SOXS资金流，也不将普通动量改名成真实再平衡流。

下一步若继续这一机制，需要取得可核验的连续历史或积累后续原始接收记录，澄清日期与衍生品敞口口径，再讨论预测模型。未建立历史数据可用性之前，不启动训练或回测。没有创建新的自动交易/采集调度，也未修改SIM。

原始两CSV、完整55行持仓、来源时刻、分钟字段审计和原生缓存已归档NAS并回读。[快照摘要](snapshot_summary.csv)、[源计划](plan.json)、[对象清单](verification_manifest.json)。负互换/非NAV语义及午夜日期不同于接收时点的合同测试通过，Notebook与原生读回通过。稳定净SOXX超额目标未完成。
